# Demo 15 - First-seen hunt: what is new versus a long baseline

**Pool:** Medium · **Visual:** new-entity bars + first-appearance scatter

**The question:** what appeared in our network this week that has never been here before?

Attacker infrastructure is new almost by definition. We take a long baseline of everything
seen historically, compare it against a short recent window, and keep only what has no
match in the baseline.

The lake makes a 90-day (or multi-year) baseline affordable, and "in A but not in B" is a
single join here rather than a nested contortion over a huge KQL result set.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `BASELINE_DAYS` - how much history counts as "known normal". 90 days is cheap in the lake
  and expensive in the analytics tier, which is rather the point of the exercise.
- `RECENT_DAYS` - the window you are hunting for novelty in.
- `ENTITY_COL` - what counts as an entity. `RemoteUrl` for domains, `RemoteIP` for
  addresses.

In [ ]:
WORKSPACE = "your-workspace-name"
BASELINE_DAYS = 90     # LENGTH of the "known normal" window, ending where RECENT_DAYS begins
RECENT_DAYS   = 7      # the window we hunt for novelty in
ENTITY_COL    = "RemoteUrl"   # try RemoteIP or RemoteUrl

## 3. Find entities in the recent window that never appear in the baseline

The hypothesis: attacker infrastructure is, almost by definition, new to you. It has not
been in your network before, because it did not exist or was not being pointed at you.

Mechanically this is a **left-anti join**: take everything from the recent window and keep
only the rows with no match in the baseline. "Show me what is in A but not in B."

That is a cheap operation over 90 days of lake data and an expensive one in the analytics
tier, which is what makes this a lake-shaped question rather than a KQL one.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

net = data_provider.read_table("DeviceNetworkEvents", WORKSPACE).filter(F.col(ENTITY_COL).isNotNull())
recent   = net.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(RECENT_DAYS)} DAYS"))
# BASELINE_DAYS is the LENGTH of the baseline, so its start sits BASELINE_DAYS before
# the recent window opens - not BASELINE_DAYS before now, which would silently shorten
# the baseline to (BASELINE_DAYS - RECENT_DAYS).
baseline = net.filter((F.col("TimeGenerated") <  F.expr(f"current_timestamp() - INTERVAL {int(RECENT_DAYS)} DAYS")) &
                      (F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(BASELINE_DAYS) + int(RECENT_DAYS)} DAYS")))

base_set = baseline.select(F.col(ENTITY_COL).alias("entity")).distinct()
new_recent = (recent.select(F.col(ENTITY_COL).alias("entity"),
                            F.col("DeviceName"), F.col("TimeGenerated"))
                    .join(base_set, "entity", "left_anti"))     # in recent, never in baseline

new_summary = (new_recent.groupBy("entity")
    .agg(F.count("*").alias("hits"), F.countDistinct("DeviceName").alias("devices"),
         F.min("TimeGenerated").alias("first_seen"))
    .orderBy(F.desc("hits"))).toPandas()
print("brand-new entities in the recent window:", len(new_summary))
new_summary.head(20)

## 4. Chart what is new

Left: the fifteen busiest brand-new entities, by hit count.
Right: when each first appeared, plotted against how much activity it generated.

**What to look for:** on the right-hand chart, a point that is both recent and high. That is
a domain which appeared from nowhere and immediately got busy, and it is the shape worth
chasing.

Expect noise. CDNs, ad networks and cloud endpoints rotate hostnames constantly and will
fill the top of this list. The technique earns its keep once you have suppressed those.

In [ ]:
if not new_summary.empty:
    top = new_summary.head(15).iloc[::-1]
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    ax[0].barh([str(e)[:40] for e in top["entity"]], top["hits"], color="#16a085")
    ax[0].set_title(f"New {ENTITY_COL} (unseen in {BASELINE_DAYS}d baseline)"); ax[0].set_xlabel("hits")
    ts = pd.to_datetime(new_summary["first_seen"])
    ax[1].scatter(ts, new_summary["hits"], alpha=.6, color="#16a085")
    ax[1].set_title("First appearance vs volume"); ax[1].set_xlabel("first seen"); ax[1].set_ylabel("hits")
    plt.tight_layout(); plt.show()
else:
    print("Nothing new - baseline already covers all recent entities.")

## Why this is a notebook hunt, not a KQL query

*'Never seen before'* is an anti-join between a long baseline and a recent window. The lake makes a 90-day (or multi-year) baseline affordable, and the left-anti join + novelty timeline is far cleaner here than nesting `not in` over huge KQL result sets.